Welcome back to the lab again. From last week, you may have worked with mass spectrometry data and processed a bit of them by yourself. This lab will look into the results that you have and we will practice a bit of data wrangling and visualization with Python. Thus, you can now understand the dataset much better with many more aspects from your result. As usual, please fill in your information here so we can give you nice some nice scores later.

- Member1:
- Member2:
- Contact email:

Same here, there will be ten questions and three bonus questions for you to answer. Please try to elaborate this exercise with the lectures from the first weeks. The main goal of this lab is that you are not afraid to work with mass spectrometry as it is amazing. Woo hoo.

## Intended learning outcomes (ILOs)

On completion of the lab, the student should be able to:

* demonstrate data-processing procedures in mass spectrometry proteomics
* demonstrate the ability to answer statistical questions with computational tools in mass spectrometry
* identify quality of high-throughput dataset and handle with statistical understandings
* identify relevant issues in technologies and data with accessible visualization techniques

## Setup

**Run this cell first.** Local users with the `py-cb2110` environment can skip; Google Colab users must run it.

In [ ]:
!pip install pandas matplotlib seaborn missingno --quiet

## Let's start!

You may recall from what we have done in the first lab. Now, we want to look at them properly. Let's start with some basic Python programming. Please copy the result file from your Lab 1 submission and update the path below.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

# File path of the result — update this to your own file if needed
path = 'data/sdrf_openms_design_msstats_in.csv'
ms_result = pd.read_csv(path)

# Data overview

Let's check how your data look. You can do it with any spreadsheet or text editor software. Python is also one of them so don't be afraid.

In [ ]:
ms_result.head()

## Q1.
**What do you see in the output file? What are the columns and the rows?**

Ans.

You can now see that it's difficult to see them clearly as there are many rows. It's hard to get an overview of the dataset. Let's dig a bit deeper and make it a little more organised. Let's now check the number of samples. These should be related to what you have with the SDRF file previously.

In [ ]:
# Number of samples and what they are
ms_result[['Reference', 'Condition']].drop_duplicates().reset_index(drop=True)

Does it look similar? Definitely, haha.

Now, look at the proteins.

## Q2.
**Let's check the peptides and proteins. How many unique proteins and peptides? What percentage of matched proteins from your library?**

In [ ]:
# Extract a table of unique proteins, peptides and precursor charge
# ms_result[['ProteinName', 'PeptideSequence', 'PrecursorCharge']].drop_duplicates()

Ans.

## Q3.
**Do you see any peptide modification during sample preparation? If yes, why do we need them?**

Ans.

## BQ1.
**Please show a summarised table containing numbers of protein counts and the detectable peptide numbers. For example, there are 10 proteins and each of them has 5 detectable peptides.**

In [ ]:
# Peptide count per protein
# Hint: groupby() and nunique()

# ms_result.groupby('ProteinName')['PeptideSequence'].nunique().reset_index()

# Dynamic range

Now, let's roughly look at the intensity of the peptides. We will use seaborn and matplotlib to visualize the data.

In [ ]:
plot_df = ms_result.copy()
plot_df['PeptideSequencePC'] = (
    plot_df['PeptideSequence'] + '_' + plot_df['PrecursorCharge'].astype(str)
)
plot_df = plot_df.sort_values('Intensity')

plt.figure(figsize=(12, 5))
sns.scatterplot(
    data=plot_df,
    x='PeptideSequencePC',
    y='Intensity',
    hue='Reference',
    alpha=0.4,
    s=15
)
plt.yscale('log')
plt.xlabel('Peptides')
plt.ylabel('Intensity')
plt.title('Overall peptide Intensity')
plt.xticks([])
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
plt.tight_layout()
plt.show()

You can now see the dynamic range of every peptide that was detected. The intensity of the peptides is quite different, maybe in some samples.

## Q4.
**What can we imply from this plot? What is the dynamic range of the dataset?**

Ans.

## Q5.
**Pick one protein that is comprised of 10 detectable peptides. Visualize the peptide intensity. Do we see every peptide in every sample? Does every peptide have the same intensity in each sample? If not, why?**

In [ ]:
# Find proteins with ~10 detectable peptides
pept_per_prot = (
    ms_result.groupby('ProteinName')['PeptideSequence']
    .nunique()
    .reset_index()
    .rename(columns={'PeptideSequence': 'n_peptides'})
)

# Show proteins with exactly 10 unique peptides
pept_per_prot[pept_per_prot['n_peptides'] == 10]

In [ ]:
# Replace with your chosen protein below
chosen_protein = 'YOUR_PROTEIN_NAME_HERE'

prot_df = ms_result[ms_result['ProteinName'] == chosen_protein].copy()
prot_df['PeptideSequencePC'] = (
    prot_df['PeptideSequence'] + '_' + prot_df['PrecursorCharge'].astype(str)
)

plt.figure(figsize=(10, 5))
sns.stripplot(
    data=prot_df,
    x='PeptideSequencePC',
    y='Intensity',
    hue='Reference',
    dodge=True,
    alpha=0.7
)
plt.yscale('log')
plt.xticks(rotation=45, ha='right')
plt.title(f'Peptide intensity — {chosen_protein}')
plt.tight_layout()
plt.show()

Ans.

# Missing data

Let's now visualize the data that you already have with their intensities at the peptide precursor level.

In [ ]:
select_pept = (
    ms_result[['Reference', 'PeptideSequence', 'PrecursorCharge', 'Intensity']]
    .pivot_table(
        index=['PeptideSequence', 'PrecursorCharge'],
        columns='Reference',
        values='Intensity'
    )
    .reset_index()
)

msno.matrix(select_pept, sparkline=False, figsize=(12, 5))
plt.title('Missing values in peptide data')
plt.tight_layout()
plt.show()

The problem now is that we can detect some missing data in the dataset. This is a common problem in mass spectrometry data. We can see that some peptides are not detected in some samples.

## Q6.
**Why are there missing values in MS?**

Ans.

Let's now select good quality peptides based on the missing data. We will remove the peptides that are not detected in more than 50% of the samples.

## Q7.
**From the `select_pept` table, remove the peptides that are not detected in more than 50% of the samples. How many unique proteins and peptides are left?**

In [ ]:
# Remove peptides missing in more than 50% of samples
sample_cols = [c for c in select_pept.columns if c not in ['PeptideSequence', 'PrecursorCharge']]
threshold = len(sample_cols) * 0.5

filter_pept = select_pept[
    select_pept[sample_cols].notna().sum(axis=1) >= threshold
][['PeptideSequence', 'PrecursorCharge']]

ms_result_filt = ms_result.merge(filter_pept, on=['PeptideSequence', 'PrecursorCharge'])

# Protein count
print("Unique proteins:", ms_result_filt['ProteinName'].nunique())
# Peptide count
print("Unique peptides:", ms_result_filt['PeptideSequence'].nunique())

Looks like we are more confident with the data now. Let's now calculate the protein abundance. The rule is we expect all peptide precursors to be detected with similar intensity in every protein. Meaning that we can average the intensity of the peptides in each protein and compare them with the rest.

## Q8.
**What are the protein abundances in each sample?**

In [ ]:
# group_by ProteinName + Reference, summarise mean Intensity
prot_level = (
    ms_result_filt
    .groupby(['ProteinName', 'Reference'])['Intensity']
    .mean()
    .reset_index()
    .rename(columns={'Intensity': 'Abundance'})
)

prot_level.head(10)

# Protein concentration

## Q9.
**Let's plot a dynamic range of protein concentration in each sample. What can we imply from this plot?**

In [ ]:
prot_sorted = prot_level.sort_values('Abundance')

plt.figure(figsize=(12, 5))
sns.scatterplot(
    data=prot_sorted,
    x='ProteinName',
    y='Abundance',
    hue='Reference',
    alpha=0.5,
    s=15
)
plt.yscale('log')
plt.xticks([])
plt.xlabel('Proteins')
plt.ylabel('Mean Intensity (Abundance)')
plt.title('Dynamic range of protein concentration per sample')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
plt.tight_layout()
plt.show()

Ans.

## Q10.
**Visualize the concentration of protein MYH9 (`sp|P35579|MYH9_HUMAN`) with a boxplot. What can we summarise here? Additionally, if it is in your experiment, how would you proceed?**

In [ ]:
# Add Condition back from the original table
prot_condition = prot_level.merge(
    ms_result[['Reference', 'Condition']].drop_duplicates(),
    on='Reference'
)

# Select MYH9
myh9 = prot_condition[prot_condition['ProteinName'] == 'sp|P35579|MYH9_HUMAN']

plt.figure(figsize=(6, 4))
sns.boxplot(data=myh9, x='Condition', y='Abundance', palette='Set2')
sns.stripplot(data=myh9, x='Condition', y='Abundance', color='black', alpha=0.6, size=5)
plt.title('MYH9 protein abundance by condition')
plt.yscale('log')
plt.tight_layout()
plt.show()

Ans.

## BQ2.
**Plot the abundance of the most differentiated protein**

In [ ]:
# Hint: calculate mean abundance per Condition for each protein,
# then find the protein with the largest fold-change between conditions

# prot_condition.groupby(['ProteinName', 'Condition'])['Abundance'].mean().unstack()

Ans.

## BQ3.
**What are the advantages and disadvantages of MS Proteomics?**

Ans.